# Raptures Analysis

## Imports

In [25]:
import pandas as pd
import ruptures as rpt
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from sklearn.preprocessing import StandardScaler

final_daily_df = pd.read_csv('processed/weighted_final_daily_df.csv', parse_dates=["date"])

print(final_daily_df.head())
print(final_daily_df.info())

        date  tweet_count  likeCount  quoteCount  retweetCount  replyCount  \
0 2015-01-01            0        NaN         NaN           NaN         NaN   
1 2015-01-02            0        NaN         NaN           NaN         NaN   
2 2015-01-03            0        NaN         NaN           NaN         NaN   
3 2015-01-04            0        NaN         NaN           NaN         NaN   
4 2015-01-05            2     3575.0         3.0        3625.0       400.0   

        neg       neu       pos  polarized  ...  learning_educational  \
0       NaN       NaN       NaN        NaN  ...                   NaN   
1       NaN       NaN       NaN        NaN  ...                   NaN   
2       NaN       NaN       NaN        NaN  ...                   NaN   
3       NaN       NaN       NaN        NaN  ...                   NaN   
4  0.022617  0.932019  0.045365        0.0  ...              0.001767   

      music  news_social_concern  other_hobbies  relationships  \
0       NaN               

In [26]:
# Plot 1: Tweet-Aktivität
fig1 = go.Figure()
fig1.add_trace(go.Scatter(x=final_daily_df["date"], y=final_daily_df['tweet_count'], mode='lines', name='Tweet Count'))

# Apply change point detection for tweet activity
tweet_activity_data = final_daily_df["tweet_count"].fillna(0).values.reshape(-1, 1)

# Use ruptures for change point detection
model = rpt.Binseg(model="l2")  # Binary segmentation with l2 norm
n_bkps = 2  # Number of breakpoints to detect
breakpoints = model.fit_predict(tweet_activity_data, n_bkps=n_bkps)

# Add vertical lines for detected breakpoints
for bkp in breakpoints[:-1]:  # Exclude the last breakpoint (end of data)
    fig1.add_vline(x=final_daily_df["date"].iloc[bkp], line=dict(color="blue", dash="dash"), name="Breakpoint")

#print breakpoints
print("Detected breakpoints for tweet activity:", breakpoints)
print("Breakpoints as dates:", final_daily_df['date'].iloc[breakpoints[:-1]].tolist())

fig1.show()

Detected breakpoints for tweet activity: [2860, 3530, 3756]
Breakpoints as dates: [Timestamp('2022-10-31 00:00:00'), Timestamp('2024-08-31 00:00:00')]


In [27]:
# Plot: Sentiment Analysis (pos, neu, neg) with Change Point Detection

fig_sent = go.Figure()
fig_sent.add_trace(go.Scatter(
    x=final_daily_df["date"],
    y=final_daily_df["pos"],
    mode='lines',
    name='Positive'
))
fig_sent.add_trace(go.Scatter(
    x=final_daily_df["date"],
    y=final_daily_df["neu"],
    mode='lines',
    name='Neutral'
))
fig_sent.add_trace(go.Scatter(
    x=final_daily_df["date"],
    y=final_daily_df["neg"],
    mode='lines',
    name='Negative'
))

# Prepare data for change point detection (all three sentiments)
sentiment_data = final_daily_df[["pos", "neu", "neg"]].fillna(0).values

model_sent = rpt.Binseg(model="l2")
n_bkps_sent = 2  # Adjust as needed
breakpoints_sent = model_sent.fit_predict(sentiment_data, n_bkps=n_bkps_sent)

# Add vertical lines for detected breakpoints
for bkp in breakpoints_sent[:-1]:
    fig_sent.add_vline(
        x=final_daily_df["date"].iloc[bkp],
        line=dict(color="blue", dash="dash"),
        name="Breakpoint"
    )

fig_sent.update_layout(
    title="Sentiment Trends (pos, neu, neg) with Change Points",
    xaxis_title="Datum",
    yaxis_title="Sentiment Score"
)

print("Detected breakpoints for sentiment (pos, neu, neg):", breakpoints_sent)
print("Breakpoints as dates:", final_daily_df['date'].iloc[breakpoints_sent[:-1]].tolist())

fig_sent.show()

Detected breakpoints for sentiment (pos, neu, neg): [1160, 2860, 3756]
Breakpoints as dates: [Timestamp('2018-03-06 00:00:00'), Timestamp('2022-10-31 00:00:00')]


In [28]:
# Plot 3: Anteil Polarisiert
fig3 = go.Figure()
fig3.add_trace(go.Scatter(x=final_daily_df["date"], y=final_daily_df['polarized'], name='Polarisiert', mode='lines'))

fig3.update_layout(title="Anteil polarisiert", xaxis_title="Datum", yaxis_title="Anteil")

# Apply change point detection for polarization trends
polarization_data = final_daily_df[["polarized"]].fillna(0).values

# Use ruptures for change point detection
model = rpt.Binseg(model="l2")  # Binary segmentation with rbm norm
n_bkps = 2  # Number of breakpoints to detect
breakpoints = model.fit_predict(polarization_data, n_bkps=n_bkps)

# Add vertical lines for detected breakpoints
for bkp in breakpoints[:-1]:  # Exclude the last breakpoint (end of data)
    fig3.add_vline(x=final_daily_df["date"].iloc[bkp], line=dict(color="blue", dash="dash"), name="Breakpoint")

#print breakpoints
print("Detected breakpoints for polarization trends:", breakpoints)
print("Breakpoints as dates:", final_daily_df['date'].iloc[breakpoints[:-1]].tolist())

fig3.show()

Detected breakpoints for polarization trends: [1150, 3485, 3756]
Breakpoints as dates: [Timestamp('2018-02-24 00:00:00'), Timestamp('2024-07-17 00:00:00')]


In [ ]:
# Plot 4: Emotionen
fig4 = go.Figure()
for col in ['anger', 'disgust', 'fear', 'joy', 'neutral', 'sadness', 'surprise']:
    fig4.add_trace(go.Scatter(x=final_daily_df["date"], y=final_daily_df[col], mode='lines', name=col.capitalize()))
fig4.update_layout(title="Emotion Scores", xaxis_title="Datum", yaxis_title="Score")

# Apply change point detection for emotion trends
emotion_data = final_daily_df[['anger', 'disgust', 'fear', 'joy', 'neutral', 'sadness', 'surprise']].fillna(0).values

# Use ruptures for change point detection
model = rpt.Binseg(model="l2")  # Binary segmentation with l2 norm
n_bkps = 2  # Number of breakpoints to detect
breakpoints = model.fit_predict(emotion_data, n_bkps=n_bkps)

# Add vertical lines for detected breakpoints
for bkp in breakpoints[:-1]:  # Exclude the last breakpoint (end of data)
    fig4.add_vline(x=final_daily_df["date"].iloc[bkp], line=dict(color="blue", dash="dash"), name="Breakpoint")

# Print breakpoints
print("Detected breakpoints for emotion trends:", breakpoints)
print("Breakpoints as dates:", final_daily_df['date'].iloc[breakpoints[:-1]].tolist())

fig4.show()

Detected breakpoints for rolling emotion trends: [455, 1190, 3756]
Breakpoints as dates: [Timestamp('2016-03-31 00:00:00'), Timestamp('2018-04-05 00:00:00')]


In [30]:
# Plot 5: Persönlichkeit
fig5 = go.Figure()
for col in ['Extroversion', 'Neuroticism', 'Agreeableness', 'Conscientiousness', 'Openness']:
    fig5.add_trace(go.Scatter(x=final_daily_df["date"], y=final_daily_df[col], mode='lines', name=col))
fig5.update_layout(title="Big Five Traits", xaxis_title="Datum", yaxis_title="Score (0–1)")

# Apply change point detection for personality traits
personality_data = final_daily_df[['Extroversion', 'Neuroticism', 'Agreeableness', 'Conscientiousness', 'Openness']].fillna(0).values

# Use ruptures for change point detection
model = rpt.Binseg(model="l2")  # Binary segmentation with l2 norm
n_bkps = 2  # Number of breakpoints to detect
breakpoints = model.fit_predict(personality_data, n_bkps=n_bkps)

# Add vertical lines for detected breakpoints
for bkp in breakpoints[:-1]:  # Exclude the last breakpoint (end of data)
    fig5.add_vline(x=final_daily_df["date"].iloc[bkp], line=dict(color="blue", dash="dash"), name="Breakpoint")

print("Detected breakpoints for personality traits:", breakpoints)
print("Breakpoints as dates:", final_daily_df['date'].iloc[breakpoints[:-1]].tolist())

fig5.show()

Detected breakpoints for personality traits: [455, 1150, 3756]
Breakpoints as dates: [Timestamp('2016-03-31 00:00:00'), Timestamp('2018-02-24 00:00:00')]


In [38]:
# === 1. Daten vorbereiten ===
# Kombiniere alle relevanten Features außer dem Datum und fülle NaN mit 0
all_features = final_daily_df.drop(columns=["date"]).fillna(0)  # shape: (n_samples, n_features)

# Skaliere die Daten (Standardisierung auf Mittelwert 0, Std 1)
scaler = StandardScaler()
features_scaled = scaler.fit_transform(all_features.values)

# === 2. Ruptures: Change Point Detection auf allen Features gemeinsam ===
model = rpt.Binseg(model="l2")  # Du kannst auch "l2" ausprobieren
n_bkps = 3  # Anzahl der Breakpoints, je nach Datensatz anpassen
model.fit(features_scaled)
joint_breakpoints = model.predict(n_bkps=n_bkps)

# === 3. Visualisierung auf Tweet Count (kann auch auf andere übertragen werden) ===
fig1 = go.Figure()

# Plot der Rolling Tweet Count Zeitreihe
fig1.add_trace(go.Scatter(
    x=final_daily_df["date"],
    y=final_daily_df["tweet_count"].rolling(window=7, min_periods=1).mean(),
    mode='lines',
    name='Tweet Count (Rolling)'
))

# Füge globale Breakpoints als vertikale Linien hinzu
for bkp in joint_breakpoints[:-1]:  # Letzten Punkt weglassen (Ende der Zeitreihe)
    fig1.add_vline(
        x=final_daily_df["date"].iloc[bkp],
        line=dict(color="green", dash="dash"),
        name="Global Breakpoint"
    )

fig1.update_layout(
    title="Tweet count with global change points",
    xaxis_title="Date",
    yaxis_title="Rolling tweet count"
)

fig1.show()

# === 4. Ausgabe der Breakpoints im Terminal ===
print("Detected joint breakpoints (indices):", joint_breakpoints)
print("Breakpoints as dates:", final_daily_df['date'].iloc[joint_breakpoints[:-1]].tolist())


# === Rolling Emotion Scores mit globalen Breakpoints ===
fig_emotion_global = go.Figure()

# Plot rolling mean for each emotion
for col in ['anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise']:
    fig_emotion_global.add_trace(go.Scatter(
        x=final_daily_df["date"],
        y=final_daily_df[col].rolling(window=100, min_periods=1).mean(),
        mode='lines',
        name=col.capitalize()
    ))

# Add global breakpoints (from joint_breakpoints) as vertical lines
for bkp in joint_breakpoints[:-1]:  # Exclude last (end of data)
    fig_emotion_global.add_vline(
        x=final_daily_df["date"].iloc[bkp],
        line=dict(color="green", dash="dash"),
        name="Global Breakpoint"
    )

fig_emotion_global.update_layout(
    title="Emotion scores (rolling 100 days) with global change points",
    xaxis_title="Date",
    yaxis_title="Score"
)

fig_emotion_global.show()

Detected joint breakpoints (indices): [1160, 2855, 3480, 3756]
Breakpoints as dates: [Timestamp('2018-03-06 00:00:00'), Timestamp('2022-10-26 00:00:00'), Timestamp('2024-07-12 00:00:00')]


In [ ]:
# Show global breakpoints on the emotion scores (rolling window)
fig_emotion_global = go.Figure()

# Plot rolling mean for each emotion
for col in ['anger', 'disgust', 'fear', 'joy', 'neutral', 'sadness', 'surprise']:
    fig_emotion_global.add_trace(go.Scatter(
        x=final_daily_df["date"],
        y=final_daily_df[col].rolling(window=window_size, min_periods=1).mean(),
        mode='lines',
        name=col.capitalize()
    ))

# Add global breakpoints (from joint_breakpoints) as vertical lines
for bkp in joint_breakpoints[:-1]:  # Exclude last (end of data)
    fig_emotion_global.add_vline(
        x=final_daily_df["date"].iloc[bkp],
        line=dict(color="green", dash="dash"),
        name="Global Breakpoint"
    )

fig_emotion_global.update_layout(
    title="Emotion scores (Rolling 7 days) with global change points",
    xaxis_title="Date",
    yaxis_title="Score"
)

fig_emotion_global.show()

In [32]:
# === Parameter ===
# Maximal gewünschte Anzahl an Breakpoints
default_n_bkps = 3

# === 1. Datenvorbereitung ===
# final_daily_df muss eine 'date'-Spalte besitzen und alle anderen Spalten sind die Features
final_daily_df['date'] = pd.to_datetime(final_daily_df['date'])
all_features = final_daily_df.drop(columns=['date']).fillna(0)

# === 2. Kategorien definieren ===
activity_cols = ['tweet_count']
engagement_cols = ['retweetCount', 'replyCount', 'likeCount', 'quoteCount']
sentiment_cols = ['pos', 'neu', 'neg']
emotion_cols = ['anger', 'disgust', 'fear', 'joy', 'neutral', 'sadness', 'surprise']
personality_cols = ['Extroversion', 'Neuroticism', 'Agreeableness', 'Conscientiousness', 'Openness']
topic_cols = [
    'arts_culture', 'business_entrepreneurs', 'celebrity_pop_culture', 'diaries_daily_life',
    'family', 'fashion_style', 'film_tv_video', 'fitness_&_health', 'food_&_dining', 'gaming',
    'learning_educational', 'music', 'news_social_concern', 'other_hobbies', 'relationships',
    'science_technology', 'sports', 'travel_adventure', 'youth_student_life'
]

categories = {
    'All Features': all_features,
    'Activity': all_features[activity_cols],
    'Engagement': all_features[engagement_cols],
    'Sentiment': all_features[sentiment_cols],
    'Emotion': all_features[emotion_cols],
    'Personality': all_features[personality_cols],
    'Topic': all_features[topic_cols]
}

# === 3. Change Point Detection (1 bis default_n_bkps) ===
breakpoints_data = {}  # dict[(category, n_bkps)] -> list of dates
scaler = StandardScaler()
for category, df_feats in categories.items():
    feats_scaled = scaler.fit_transform(df_feats.values)
    for n_bkps in range(1, default_n_bkps + 1):
        model = rpt.Binseg(model='l2').fit(feats_scaled)
        bkps = model.predict(n_bkps=n_bkps)
        change_idxs = bkps[:-1]
        dates = final_daily_df['date'].iloc[[idx-1 for idx in change_idxs]].dt.strftime('%Y-%m-%d').tolist()
        breakpoints_data[(category, n_bkps)] = dates

# === 4a. Visualisierung 1: nach Kategorie (innerhalb: verschiedene BP-Zahlen) ===
fig1 = go.Figure()
y_labels1 = []
y_pos1 = 0
for category in categories.keys():
    for n_bkps in range(1, default_n_bkps + 1):
        label = f"{category} ({n_bkps} BP)"
        y_labels1.append(label)
        for dt in breakpoints_data.get((category, n_bkps), []):
            fig1.add_trace(go.Scatter(
                x=[pd.to_datetime(dt)],
                y=[y_pos1],
                mode='markers',
                marker=dict(size=10, color='blue'),
                hovertemplate=f"{label}<br>Date: {dt}<extra></extra>",
                showlegend=False
            ))
        y_pos1 += 1
fig1.update_layout(
    title="Change Points nach Kategorie und Breakpoint-Anzahl",
    xaxis_title="Date",
    yaxis=dict(
        tickmode='array',
        tickvals=list(range(len(y_labels1))),
        ticktext=y_labels1
    ),
    height=800,
    showlegend=False
)
fig1.show()

# === 4b. Visualisierung 2: nach Breakpoint-Anzahl (innerhalb: verschiedene Kategorien) ===
fig2 = go.Figure()
y_labels2 = []
y_pos2 = 0
for n_bkps in range(1, default_n_bkps + 1):
    for category in categories.keys():
        label = f"{category} ({n_bkps} BP)"
        y_labels2.append(label)
        for dt in breakpoints_data.get((category, n_bkps), []):
            fig2.add_trace(go.Scatter(
                x=[pd.to_datetime(dt)],
                y=[y_pos2],
                mode='markers',
                marker=dict(size=10, color='orange'),
                hovertemplate=f"{label}<br>Date: {dt}<extra></extra>",
                showlegend=False
            ))
        y_pos2 += 1
fig2.update_layout(
    title="Change Points nach Breakpoint-Anzahl und Kategorie",
    xaxis_title="Date",
    yaxis=dict(
        tickmode='array',
        tickvals=list(range(len(y_labels2))),
        ticktext=y_labels2
    ),
    height=800,
    showlegend=False
)
fig2.show()